In [20]:
import os,sys,requests,zipfile,shutil;
from osgeo import gdal,ogr,osr;
from osgeo_utils import gdal_calc,gdal_edit;
gdal.UseExceptions();
g_cache_gb = 96;
g_cpus     = 'ALL_CPUS';

sys.path.append(os.path.join(os.path.expanduser('~'),'notebooks'));
import common;

lddk = os.path.join(os.path.expanduser('~'),'loading_dock','grids_2026','usgsdem');
if not os.path.exists(lddk):
    os.mkdir(lddk);

aoi_gpkg = os.path.join(lddk,'aoi.gpkg');
outname = 'va_buff';


In [21]:
fetch_aoi = False;

if fetch_aoi is True:

    if os.path.exists(aoi_gpkg):
        os.remove(aoi_gpkg);

    srs = osr.SpatialReference();
    srs.ImportFromEPSG(5070);

    drv = ogr.GetDriverByName('GPKG');
    dst_ds = drv.CreateDataSource(aoi_gpkg);
    
    dst_lyr1 = dst_ds.CreateLayer('aoi',geom_type=ogr.wkbPolygon,srs=srs);
    nf1 = ogr.FieldDefn('aoi_id',ogr.OFTString);
    nf1.SetWidth(255);
    dst_lyr1.CreateField(nf1);    
    dst_ds = None;

    qry = '?outSR=5070&where=STUSAB%3D%27VA%27&outFields=STUSAB&f=json';
    gdal.VectorTranslate(
         srcDS            = r'https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/State_County/MapServer/0/query' + qry
        ,destNameOrDestDS = aoi_gpkg
        ,layerName        = 'aoi'
        ,format           = 'GPKG'
        ,dstSRS           = 'EPSG:5070'
        ,accessMode       = 'append'
    );

    ds = drv.Open(aoi_gpkg,1);
    ds.ExecuteSQL("UPDATE aoi SET aoi_id = 'VA'",dialect='SQLITE');
    lyr = ds.GetLayer('aoi');
    for feat in lyr:
        geom = feat.GetGeometryRef();
        if geom:
            buff_geom = geom.Buffer(8046.72);
            feat.SetGeometry(buff_geom);
            lyr.SetFeature(feat);

    ds = None;            
    

In [22]:
dems30 = [
     ['n37w076','USGS_1_n37w076_20260324.tif']
    ,['n37w077','USGS_1_n37w077_20260324.tif']
    ,['n37w078','USGS_1_n37w078_20250507.tif']
    ,['n37w079','USGS_1_n37w079_20250507.tif']
    ,['n37w080','USGS_1_n37w080_20210305.tif']
    ,['n37w081','USGS_1_n37w081_20240611.tif']
    ,['n37w082','USGS_1_n37w082_20240611.tif']
    ,['n37w083','USGS_1_n37w083_20240923.tif']
    ,['n37w084','USGS_1_n37w084_20240923.tif']

    ,['n38w076','USGS_1_n38w076_20260324.tif']
    ,['n38w077','USGS_1_n38w077_20260324.tif']
    ,['n38w078','USGS_1_n38w078_20211220.tif']
    ,['n38w079','USGS_1_n38w079_20211229.tif']
    ,['n38w080','USGS_1_n38w080_20211229.tif']
    ,['n38w081','USGS_1_n38w081_20230816.tif']
    ,['n38w082','USGS_1_n38w082_20240923.tif']
    ,['n38w083','USGS_1_n38w083_20240923.tif']
    ,['n38w084','USGS_1_n38w084_20260227.tif']

    ,['n39w075','USGS_1_n39w075_20210624.tif']
    ,['n39w076','USGS_1_n39w076_20260324.tif']
    ,['n39w077','USGS_1_n39w077_20260407.tif']
    ,['n39w078','USGS_1_n39w078_20260407.tif']
    ,['n39w079','USGS_1_n39w079_20250512.tif']
    ,['n39w080','USGS_1_n39w080_20230816.tif']
    ,['n39w081','USGS_1_n39w081_20230816.tif']

    ,['n40w078','USGS_1_n40w078_20260407.tif']
    ,['n40w079','USGS_1_n40w079_20230816.tif']
];

dems10 = [
     ['n37w076','USGS_13_n37w076_20260324.tif']
    ,['n37w077','USGS_13_n37w077_20260324.tif']
    ,['n37w078','USGS_13_n37w078_20250507.tif']
    ,['n37w079','USGS_13_n37w079_20250507.tif']
    ,['n37w080','USGS_13_n37w080_20210305.tif']
    ,['n37w081','USGS_13_n37w081_20240611.tif']
    ,['n37w082','USGS_13_n37w082_20240611.tif']
    ,['n37w083','USGS_13_n37w083_20240923.tif']
    ,['n37w084','USGS_13_n37w084_20240923.tif']

    ,['n38w076','USGS_13_n38w076_20260324.tif']
    ,['n38w077','USGS_13_n38w077_20260324.tif']
    ,['n38w078','USGS_13_n38w078_20211220.tif']
    ,['n38w079','USGS_13_n38w079_20211229.tif']
    ,['n38w080','USGS_13_n38w080_20211229.tif']
    ,['n38w081','USGS_13_n38w081_20230816.tif']
    ,['n38w082','USGS_13_n38w082_20230816.tif']
    ,['n38w083','USGS_13_n38w083_20240923.tif']
    ,['n38w084','USGS_13_n38w084_20240923.tif']

    ,['n39w075','USGS_13_n39w075_20210624.tif']
    ,['n39w076','USGS_13_n39w076_20260324.tif']
    ,['n39w077','USGS_13_n39w077_20260407.tif']
    ,['n39w078','USGS_13_n39w078_20260407.tif']
    ,['n39w079','USGS_13_n39w079_20250512.tif']
    ,['n39w080','USGS_13_n39w080_20230816.tif']
    ,['n39w081','USGS_13_n39w081_20230816.tif']

    ,['n40w078','USGS_13_n40w078_20260407.tif']
    ,['n40w079','USGS_13_n40w079_20230816.tif']
];


In [23]:
download_dems = False;

if download_dems is True:
    url30 = r'https://prd-tnm.s3.amazonaws.com/StagedProducts/Elevation/1/TIFF/historical/';
    url10 = r'https://prd-tnm.s3.amazonaws.com/StagedProducts/Elevation/13/TIFF/historical/';

    for item in dems10:
        u10 = url10 + item[0] + '/' + item[1];
        
        #print('   checking ' + u10);
        #common.check_link(u10);
        
        print('downloading ' + u10);
        common.download_link(u10,os.path.join(lddk,item[1]));
        

In [31]:
# Step to project tiffs to 5070

wash_tiffs = True;

if wash_tiffs is True:
    gdal.SetCacheMax(g_cache_gb * 1024 * 1024 * 1024);
    gdal.SetConfigOption('GDAL_NUM_THREADS',g_cpus);

    for item in dems10:
        print("washing " + item[1]);
        
        in10  = os.path.join(lddk,item[1]);
        out10 = os.path.join(lddk,item[1].replace('.tif','_wash.tif'));

        if os.path.exists(out10):
            os.remove(out10);

        options = gdal.WarpOptions(
             format          = 'GTiff'
            ,xRes            = 10
            ,yRes            = 10
            ,resampleAlg     ='near'
            ,creationOptions = ['COMPRESS=DEFLATE']
            ,dstSRS          = 'EPSG:5070'
        );
        gdal.Warp(
             srcDSOrSrcDSTab  = in10
            ,destNameOrDestDS = out10
            ,options          = options
        );
        

washing USGS_13_n37w076_20260324.tif


washing USGS_13_n37w077_20260324.tif


washing USGS_13_n37w078_20250507.tif


washing USGS_13_n37w079_20250507.tif


washing USGS_13_n37w080_20210305.tif


...

washing USGS_13_n39w079_20250512.tif


washing USGS_13_n39w080_20230816.tif


washing USGS_13_n39w081_20230816.tif


washing USGS_13_n40w078_20260407.tif


washing USGS_13_n40w079_20230816.tif


In [32]:
# Boolean flag to build gridset VRTs 

build_vrt = True;

if build_vrt is True:

    vrt10 = os.path.join(lddk,'usgsdem_10m_wash.vrt');
    lst10 = os.path.join(lddk,'usgsdem_10m_wash_list.txt');

    if os.path.exists(vrt10):
        os.remove(vrt10);

    if os.path.exists(lst10):
        os.remove(lst10);

    ary10 = [];
    with open(lst10,"w") as f:
        for item in dems10:
            t10  = os.path.join(lddk,item[1].replace('.tif','_wash.tif'));
            f.write(t10 + '\n');
            ary10.append(t10);

    o10 = gdal.BuildVRTOptions(
         xRes = 10
        ,yRes = 10
        ,srcNodata = -999999
        ,VRTNodata = -999999
    );
    v10 = gdal.BuildVRT(
         destName        = vrt10
        ,srcDSOrSrcDSTab = ary10
        ,options         = o10
    );
    v10 = None;


In [34]:
# Boolean flag to mosaic and clip to area of interest

clip_to_aoi = True;

if clip_to_aoi is True:
    gdal.SetCacheMax(g_cache_gb * 1024 * 1024 * 1024);
    gdal.SetConfigOption('GDAL_NUM_THREADS','ALL_CPUS');
    
    vrt10 = os.path.join(lddk,'usgsdem_10m_wash.vrt');
    clp10 = os.path.join(lddk,'usgsdem_10m_wash.tif');

    if os.path.exists(clp10):
        os.remove(clp10);
        
    o10 = gdal.WarpOptions(
         format           = 'GTiff'
        ,cutlineDSName    = aoi_gpkg
        ,cutlineLayer     = 'aoi'
        ,cropToCutline    = True
        ,dstNodata        = -999999
        ,creationOptions  = ['TFW=YES','COMPRESS=DEFLATE','TILED=YES','BIGTIFF=YES']
    );
    gdal.Warp(
         srcDSOrSrcDSTab  = vrt10
        ,destNameOrDestDS = clp10
        ,options          = o10
    );
        